# Synapse vs Databricks Data Validation

Reusable notebook that compares tables between Synapse and Databricks.
Define your table mappings below, then run all cells to get a full comparison report.

**Requirements:** `pyodbc`, `polars`, `databricks-sql-connector`

In [15]:
import sys
import os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), ''))
sys.path.insert(0, os.path.abspath('..'))

import pyodbc
from databricks import sql
import polars as pl

from validation_utils import compare_synapse_vs_databricks

## Configuration

Edit the table mappings and connection details below.

In [16]:
# ─── Table Mappings ─────────────────────────────────────────────────────────────
# Dict of source_system → list of (synapse_schema, synapse_table, dbx_catalog, dbx_schema, dbx_table)
# Synapse source: dal_v views (current data only, no OMD filters needed)

TABLE_MAPPINGS = {
    "IOC_MMRS": [
        ("dal_v", "vw_UDLMMRSIOC_mmrs_cs_vw_cycle", "rtio_tactical_sourcealigned", "ioc_mmrs", "mmrs_vw_cycle"),
        ("dal_v", "vw_UDLMMRSIOC_mmrs_cs_vw_event", "rtio_tactical_sourcealigned", "ioc_mmrs", "mmrs_vw_event"),
        ("dal_v", "vw_UDLMMRSIOC_vw_location", "rtio_tactical_sourcealigned", "ioc_mmrs", "mmrs_vw_location"),
        ("dal_v", "vw_UDLMMRSIOC_vw_rf_material", "rtio_tactical_sourcealigned", "ioc_mmrs", "mmrs_vw_rf_material"),
        ("dal_v", "vw_UDLMMRSIOC_vw_rf_timecode", "rtio_tactical_sourcealigned", "ioc_mmrs", "mmrs_vw_rf_timecode"),
        ("dal_v", "vw_UDLMMRSIOC_vw_rf_unit", "rtio_tactical_sourcealigned", "ioc_mmrs", "mmrs_vw_rf_unit"),
        ("dal_v", "vw_UDLMMRSIOC_vw_unit_fleet_fleettype", "rtio_tactical_sourcealigned", "ioc_mmrs", "mmrs_vw_unit_fleet_fleettype"),
    ],
    "GENSD03_SICOM": [
        ("dal_v", "VW_UDLGENSD03_DBO_HSP_SUIVIETATEQUIPEMENTROWNUMBER_VIEW", "_ogr_azr_syd", "gensd03_sicom", "dbo_hsp_suivietatequipementrownumber_view"),
        ("dal_v", "VW_UDLGENSD03_DBO_GEN_CODEIPT_DEF", "_ogr_azr_syd", "gensd03_sicom", "dbo_gen_codeipt_def"),
        ("dal_v", "VW_UDLGENSD03_DBO_HSP_SUIVIETATEQUIPEMENT", "_ogr_azr_syd", "gensd03_sicom", "dbo_hsp_suivietatequipement"),
        ("dal_v", "VW_UDLGENSD03_DBO_HSP_GROUPEEQUIPEMENT_DEF", "_ogr_azr_syd", "gensd03_sicom", "dbo_hsp_groupeequipement_def"),
        ("dal_v", "VW_UDLGENSD03_DBO_HSP_MODELEEQUIPEMENT_DEF", "_ogr_azr_syd", "gensd03_sicom", "dbo_hsp_modeleequipement_def"),
        ("dal_v", "VW_UDLGENSD03_DBO_HSP_EQUIPEMENT_DEF", "_ogr_azr_syd", "gensd03_sicom", "dbo_hsp_equipement_def"),
    ],
    "PI_SYSTEMS": [
        ("dal_v", "vw_udlpisystems_opsdata_mis_rtit_headgrade", "lakehouse", "analytics_and_enablement_staging_pi_systems", "opsdata_mis_rtit_headgrade"),
        ("dal_v", "vw_udlpisystems_opsdata_cs_mis_rtit_sandmetrics", "lakehouse", "analytics_and_enablement_staging_pi_systems", "opsdata_mis_rtit_sandmetrics"),
        ("dal_v", "vw_udlpisystems_opsdata_cs_pm_reporting_data", "lakehouse", "analytics_and_enablement_staging_pi_systems", "opsdata_pm_reporting_data"),
        ("dal_v", "vw_udlpisystems_opsdata_cs_pm_reporting_recovery_data", "lakehouse", "analytics_and_enablement_staging_pi_systems", "opsdata_pm_reporting_recovery_data"),
        ("dal_v", "vw_udlpisystems_pm_reporting_locations_hierarchy", "lakehouse", "analytics_and_enablement_staging_pi_systems", "globalpi_pm_reporting_locations_hierarchy"),
    ],
    "ODP": [
# dbx        ("hstg_v", "vw_hstg_UDLDWBDP_DAL_CS_FACT_HMETIMEUSAGEEVENT", "ogr_sourcealigned", "odp", "dal_fact_hmetimeusageevent"),
# dbx       ("hstg_v", "vw_hstg_UDLDWBDP_DAL_DIM_HMEASSET", "ogr_sourcealigned", "odp", "dal_dim_hmeasset"),
# syp       ("hstg_v", "vw_hstg_UDLDWBDP_DAL_DIM_HMETIMEUSAGEMODEL", "ogr_sourcealigned", "odp", "dal_dim_hmetimeusagemodel"),
# syp       ("hstg_v", "vw_hstg_UDLDWBDP_DAL_DIM_LOCATION", "ogr_sourcealigned", "odp", "dal_dim_location"),
# syp       ("hstg_v", "vw_hstg_UDLDWBDP_DAL_DIM_PAYLOADMANAGEMENT", "ogr_sourcealigned", "odp", "dal_dim_payloadmanagement"),
# dbx       ("hstg_v", "vw_hstg_UDLDWBDP_DAL_FACT_HMELOADHAULCYCLE", "ogr_sourcealigned", "odp", "dal_fact_hmeloadhaulcycle"),
        ("dal_v", "vw_UDLODP_DAL_DIM_FIXEDPLANTMETRIC", "ogr_sourcealigned", "odp", "dal_dim_fixedplantmetric"),
        ("dal_v", "vw_UDLODP_DAL_DIM_MATERIAL", "ogr_sourcealigned", "odp", "dal_dim_material"),
        ("dal_v", "vw_UDLODP_DAL_DIM_SHIFT", "ogr_sourcealigned", "odp", "dal_dim_shift"),
        ("dal_v", "vw_UDLODP_DAL_DIM_SITE", "ogr_sourcealigned", "odp", "dal_dim_site"),
        ("dal_v", "vw_UDLODP_DAL_DIM_TARGET", "ogr_sourcealigned", "odp", "dal_dim_target"),
        ("dal_v", "vw_UDLODP_DAL_FACT_FIXEDPLANTPRODUCTIONSHIFT", "ogr_sourcealigned", "odp", "dal_fact_fixedplantproductionshift"),
        ("dal_v", "vw_UDLODP_DAL_FACT_FIXEDPLANTTARGET", "ogr_sourcealigned", "odp", "dal_fact_fixedplanttarget"),
    ],
    "EGRP_GREPORTS": [
        ("dal_v", "vw_UDLRIOTINTOSYDNEYINTERNALSAPB4P1B4PSAPB4P_B4P_G42413A_ZBB_QUALITY_LOCAL_GI_GOVERNED", "ent_sourcealigned", "egrp_greports", "g42413a_zbb_quality_local_gi_governed"),
        ("dal_v", "vw_UDLRIOTINTOSYDNEYINTERNALSAPB4P1B4PSAPB4P_B4P_G44561A_SCHEDULE_COMPLIANCE_GI_GOVERNED", "ent_sourcealigned", "egrp_greports", "g44561a_schedule_compliance_gi_governed"),
        ("dal_v", "vw_UDLRIOTINTOSYDNEYINTERNALSAPB4P1B4PSAPB4P_B4P_G44561A_SCHEDULE_WORK_GI_GOVERNED", "ent_sourcealigned", "egrp_greports", "g44561a_schedule_work_gi_governed"),
        ("dal_v", "vw_UDLRIOTINTOSYDNEYINTERNALSAPB4P1B4PSAPB4P_B4P_G44564A_SNAPSHOT_BACKLOG_AGE_GOVERNED", "ent_sourcealigned", "egrp_greports", "g44564a_snapshot_backlog_age_governed"),
        ("dal_v", "vw_UDLRIOTINTOSYDNEYINTERNALSAPB4P1B4PSAPB4P_B4P_G44565A_USE_OF_RESOURCE_AVAILABILITY_GI_GOVERNED", "ent_sourcealigned", "egrp_greports", "g44565a_use_of_resource_availability_gi_governed"),
        ("dal_v", "vw_UDLRIOTINTOSYDNEYINTERNALSAPB4P1B4PSAPB4P_B4P_G44567A_PRIMARY_MAINTENANCE_CALL_COMPLIANCE_GOVERNED", "ent_sourcealigned", "egrp_greports", "g44567a_primary_maintenance_call_compliance_governed"),
        ("dal_v", "vw_UDLRIOTINTOSYDNEYINTERNALSAPB4P1B4PSAPB4P_B4P_G44569A_MAINTENANCE_PLANS_FUNCTIONAL_LOCATION_CRI_LOOKUP", "ent_sourcealigned", "egrp_greports", "g44569a_maintenance_plans_functional_location_cri_lookup"),
        ("dal_v", "vw_UDLRIOTINTOSYDNEYINTERNALSAPB4P1B4PSAPB4P_B4P_G44576A_WORK_EVENT_PROFILE_GOVERNED", "ent_sourcealigned", "egrp_greports", "g44576a_work_event_profile_governed"),
        ("dal_v", "vw_UDLRIOTINTOSYDNEYINTERNALSAPB4P1B4PSAPB4P_B4P_G52002A_PRODUCT_QUALITY", "ent_sourcealigned", "egrp_greports", "g52002a_product_quality"),
        ("dal_v", "vw_UDLRIOTINTOSYDNEYINTERNALSAPB4P1B4PSAPB4P_B4P_G52004A_TOTAL_MATERIAL_MOVED_ACTUAL", "ent_sourcealigned", "egrp_greports", "g52004a_total_material_moved_actual"),
        ("dal_v", "vw_UDLRIOTINTOSYDNEYINTERNALSAPB4P1B4PSAPB4P_B4P_G70301A_MAT_MVMT_ANLYS_PRODFACTSHEET_GOV", "ent_sourcealigned", "egrp_greports", "g70301a_mat_mvmt_anlys_prodfactsheet_gov"),
    ],
    "EGRP_IAP_STAGING": [
        ("dal_v", "vw_UDLANALYTICSANDENABLEMENTREPOSITORYAZUREIAPSTAGING_EGR_MP_RDL_DIM_OPERATING_RESPONSIBILITY", "lakehouse", "analytics_and_enablement_repository_azure_iap_staging", "egr_mp_rdl_dim_operating_responsibility"),
        ("dal_v", "vw_UDLANALYTICSANDENABLEMENTREPOSITORYAZUREIAPSTAGING_EGR_MPA_ODL_DIM_FUNCTIONAL_LOCATION", "lakehouse", "analytics_and_enablement_repository_azure_iap_staging", "egr_mpa_odl_dim_functional_location"),
# syp        ("hstg_v", "vw_hstg_UDLANALYTICSANDENABLEMENTREPOSITORYAZUREIAPSTAGING_EGR_MMP_RDL_DIM_PROCESS_HIER", "lakehouse", "analytics_and_enablement_repository_azure_iap_staging", "egr_mmp_rdl_dim_process_hier"),
# syp       ("hstg_v", "vw_hstg_UDLANALYTICSANDENABLEMENTREPOSITORYAZUREIAPSTAGING_EGR_MP_RDL_DIM_OPERATING_RESPONSIBILITY_HIER", "lakehouse", "analytics_and_enablement_repository_azure_iap_staging", "egr_mp_rdl_dim_operating_responsibility_hier"),
# syp       ("hstg_v", "vw_hstg_UDLANALYTICSANDENABLEMENTREPOSITORYAZUREIAPSTAGING_EGR_MPA_RDL_DIM_ASSET", "lakehouse", "analytics_and_enablement_repository_azure_iap_staging", "egr_mpa_rdl_dim_asset"),
    ],
    "EGRP_SAP_BRONZE": [
        ("dal_v", "vw_UDLLIBECPRTP_SAP_CS_QMFE", "ent_sourcealigned", "egrp_sap_bronze", "qmfe"),
        ("dal_v", "vw_UDLLIBECPRTP_SAP_MAKT", "lakehouse", "lib_ecp_rtp", "sap_makt"),
    ],
}

In [17]:
# ─── Connection Setup ──────────────────────────────────────────────────────────

# Synapse
synapse_server = 'syn-rt-prd-unity.sql.azuresynapse.net'
synapse_database = 'syndwrtprdunity'
synapse_username = 'byambadorjme@riotinto.com'
synapse_driver = '{ODBC Driver 17 for SQL Server}'

synapse_connection_string = f"""
    DRIVER={synapse_driver};
    SERVER={synapse_server};
    DATABASE={synapse_database};
    UID={synapse_username};
    Authentication=ActiveDirectoryInteractive;
    Encrypt=yes;
    TrustServerCertificate=no;
    Connection Timeout=30;
"""

# Databricks
databricks_access_token = os.environ["DATABRICKS_TOKEN"]
databricks_server_hostname = 'adb-2439498039309203.3.azuredatabricks.net'
databricks_http_path = '/sql/1.0/warehouses/e7d134c4348ac8b5'

# ─── Connect ──────────────────────────────────────────────────────────────────
print("Connecting to Synapse (browser auth prompt may appear)...")
synapse_conn = pyodbc.connect(synapse_connection_string)
print("✓ Synapse connected")

databricks_conn = sql.connect(
    server_hostname=databricks_server_hostname,
    http_path=databricks_http_path,
    access_token=databricks_access_token,
)
print("✓ Databricks connected")

Connecting to Synapse (browser auth prompt may appear)...
✓ Synapse connected
✓ Databricks connected


In [18]:
# ─── Display settings ─────────────────────────────────────────────────────────
pl.Config.set_tbl_rows(100)
pl.Config.set_tbl_cols(50)
pl.Config.set_fmt_str_lengths(100)

polars.config.Config

## Run Comparison

Iterates over all table mappings and produces a summary report.

In [19]:
def run_comparison(source_system: str, mappings: list, synapse_conn, databricks_conn) -> list:
    """Run comparisons for a list of table mappings and return results."""
    print(f"\n{'█' * 80}")
    print(f"  SOURCE SYSTEM: {source_system} ({len(mappings)} tables)")
    print(f"{'█' * 80}")

    results = []
    for i, (syn_schema, syn_table, dbx_catalog, dbx_schema, dbx_table) in enumerate(mappings, 1):
        print(f"\n  [{i}/{len(mappings)}] {dbx_catalog}.{dbx_schema}.{dbx_table}")
        try:
            result = compare_synapse_vs_databricks(
                databricks_conn=databricks_conn, synapse_conn=synapse_conn,
                synapse_schema_name=syn_schema, synapse_table_name=syn_table,
                databricks_catalog_name=dbx_catalog, databricks_schema_name=dbx_schema,
                databricks_table_name=dbx_table,
            )
            result["source_system"] = source_system
            result["synapse_table"] = syn_table
            result["databricks_table"] = dbx_table
            results.append(result)

            match_icon = "✓" if result["row_count_matches"] else "✗"
            print(f"    Schema: {'✓' if result['schema_matches'] else '✗'} | "
                  f"Rows: {match_icon} Syn={result['synapse_row_count']:,} Dbx={result['databricks_row_count']:,} "
                  f"(diff={result['row_diff']:+,}, {result['row_diff_pct']:+.1f}%)")
        except Exception as e:
            print(f"    ✗ ERROR: {e}")
            results.append({"source_system": source_system, "synapse_table": syn_table,
                            "databricks_table": dbx_table, "error": str(e)})

    print(f"\n  Done: {len(results)} tables compared")
    return results

In [ ]:
results = run_comparison("EGRP_IAP_STAGING", TABLE_MAPPINGS["EGRP_IAP_STAGING"], synapse_conn, databricks_conn)


████████████████████████████████████████████████████████████████████████████████
  SOURCE SYSTEM: EGRP_IAP_STAGING (2 tables)
████████████████████████████████████████████████████████████████████████████████

  [1/2] lakehouse.analytics_and_enablement_repository_azure_iap_staging.egr_mp_rdl_dim_operating_responsibility
    ✗ ERROR: ('42000', '[42000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Some part of your SQL statement is nested too deeply. Rewrite the query or break it up into smaller queries. (102043) (SQLExecDirectW)')

  [2/2] lakehouse.analytics_and_enablement_repository_azure_iap_staging.egr_mpa_odl_dim_functional_location


In [12]:
results[1]['schema_comparison_df']

column_name,synapse_data_type,column_name_right,databricks_data_type,synapse_expected_type
str,str,str,str,str
"""FLOC__MASTER_DATA_KEY_SOURCE""","""VARCHAR""","""FLOC__MASTER_DATA_KEY_SOURCE""","""STRING""","""VARCHAR"""
"""FLOC__ACQUISITION_DATE""","""DATETIME2""","""FLOC__ACQUISITION_DATE""","""DATE""","""DATE"""
"""FLOC__ACQUISITION_DATE__KEY_""","""INT""","""FLOC__ACQUISITION_DATE__KEY_""","""INT""","""INT"""
"""FLOC__ACQUISITION_VALUE_CURRENCY""","""VARCHAR""","""FLOC__ACQUISITION_VALUE_CURRENCY""","""STRING""","""VARCHAR"""
"""FLOC__ALTERNATE_FUNCTIONAL_LOCATION""","""VARCHAR""","""FLOC__ALTERNATE_FUNCTIONAL_LOCATION""","""STRING""","""VARCHAR"""
"""FLOC__ALTERNATIVE_LABELING""","""VARCHAR""","""FLOC__ALTERNATIVE_LABELING""","""STRING""","""VARCHAR"""
"""FLOC__ASSEMBLY""","""VARCHAR""","""FLOC__ASSEMBLY""","""STRING""","""VARCHAR"""
"""FLOC__ASSEMBLY__KEY_""","""VARCHAR""","""FLOC__ASSEMBLY__KEY_""","""STRING""","""VARCHAR"""
"""FLOC__ASSET__FUNCTIONAL_LOCATION_""","""VARCHAR""","""FLOC__ASSET__FUNCTIONAL_LOCATION_""","""STRING""","""VARCHAR"""


In [ ]:
# ─── Summary Report ───────────────────────────────────────────────────────────
summary_rows = []
for source_system, results in all_results.items():
    for r in results:
        if "error" in r:
            summary_rows.append({
                "source_system": source_system,
                "table": r["databricks_table"],
                "synapse_rows": None,
                "databricks_rows": None,
                "row_diff": None,
                "row_diff_pct": None,
                "schema_match": None,
                "row_count_match": None,
                "common_cols": None,
                "status": f"ERROR: {r['error'][:60]}",
            })
        else:
            status = "✓ Match" if r["row_count_matches"] and r["schema_matches"] else (
                "⚠ Row count mismatch" if not r["row_count_matches"] else "⚠ Schema mismatch"
            )
            summary_rows.append({
                "source_system": source_system,
                "table": r["databricks_table"],
                "synapse_rows": r["synapse_row_count"],
                "databricks_rows": r["databricks_row_count"],
                "row_diff": r["row_diff"],
                "row_diff_pct": r["row_diff_pct"],
                "schema_match": r["schema_matches"],
                "row_count_match": r["row_count_matches"],
                "common_cols": len(r["common_columns"]),
                "status": status,
            })

summary_df = pl.DataFrame(summary_rows)
print("\n" + "=" * 80)
print("VALIDATION SUMMARY")
print("=" * 80)
print(summary_df)

In [ ]:
# ─── Detailed Results (per table) ─────────────────────────────────────────────
# Uncomment a table index to inspect its detailed results

# idx = 0  # Change index to inspect a specific table
# r = all_results['GENSD03_SICOM'][idx]
# print(f"Table: {r['databricks_table']}")
# print(f"\nSchema Comparison:")
# display(r['schema_comparison_df'])
# print(f"\nColumn Stats Comparison:")
# display(r['stats_comparison_df'])

In [ ]:
# ─── Cleanup ──────────────────────────────────────────────────────────────────
synapse_conn.close()
databricks_conn.close()
print("Connections closed.")